# HRNet-W32: local linear head

Train a five-logit linear classifier on frozen ImageNet-1k HRNet-W32 features for the stored fold-0 split. Save feature shards, the latest training state, the selected head, and per-label metrics under `artifacts/hrnet_head/`. This stage supplies the head initialization for later Colab fine-tuning.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import torch

RUN_ID = os.environ.get("HRNET_RUN_ID", "hrnet_w32_head_fold0_seed42")
BATCH_SIZE = int(os.environ.get("HRNET_BATCH_SIZE", "2"))
HEAD_EPOCHS = int(os.environ.get("HRNET_HEAD_EPOCHS", "8"))

start = Path(os.environ.get("STAGE2_ROOT", Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / "folds.csv").is_file() and (path / "stage2" / "hrnet_head.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Set STAGE2_ROOT to the repository containing folds.csv and stage2/hrnet_head.py")
sys.path.insert(0, str(ROOT))
from stage2.hrnet_head import HeadConfig, prepare_run, repository_root, run_head, seed_everything
from stage2.experiment import LABEL_COLUMNS

config = HeadConfig(run_id=RUN_ID, batch_size=BATCH_SIZE, head_epochs=HEAD_EPOCHS)
seed_everything(config.seed)
print(f"{config.model_name} | {config.image_size}px | fold {config.validation_fold} | device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")


## Check the split and source files

The stored fold manifest is checked against the original seed-42 split. The run directory contains hashes of the labels and folds, so a restart cannot silently attach a different split to existing features.


In [ ]:
run_dir, training_rows, validation_rows = prepare_run(ROOT, config)
train_counts = {label: sum(int(row[label]) for row in training_rows) for label in LABEL_COLUMNS}
val_counts = {label: sum(int(row[label]) for row in validation_rows) for label in LABEL_COLUMNS}
print(f"Train: {len(training_rows)} | validation: {len(validation_rows)}")
print("Train positives:", train_counts)
print("Validation positives:", val_counts)
print("Artifacts:", run_dir)


## Extract frozen features and train the head

`timm` loads `hrnet_w32.ms_in1k` with its ImageNet-1k weights and normalization. Images use the existing 384-pixel letterbox; horizontal flips are extracted as a second training view. Each 64-image feature shard is written atomically and checked on restart. The head uses unweighted BCEWithLogitsLoss. After every epoch, the run writes `head_latest.pt`, `head_history.csv`, and the best head when macro F1 improves at the fixed 0.25 threshold. W&B uses the ignored `.env` key if present; local metrics remain available if W&B is offline.

On CPU, this cell may take a long time to finish feature extraction. It can be interrupted and rerun with the same run ID.


In [ ]:
run_dir = run_head(ROOT, config)


## Review the selected head

The selected head is an initialization for joint fine-tuning. Fold-0 scores select its epoch and are therefore screening scores, not an independent final estimate.


In [ ]:
import csv
import json

best = json.loads((run_dir / "head_best.json").read_text())
with (run_dir / "head_history.csv").open(newline="") as stream:
    history = list(csv.DictReader(stream))
print(f"Best epoch {best['epoch']}: macro F1 {best['macro_f1']:.5f}; val BCE {best['val_loss']:.5f}")
print("Per-label F1:", best["metrics"]["per_label_f1"])
print(f"Macro AP: {best['metrics']['macro_ap']:.5f}; per-label AP:", best["metrics"]["per_label_ap"])
print("Saved:", run_dir / "head_best.pt", "and", run_dir / "head_latest.pt")


## Conclusion

Compare the full per-label fold-0 metrics and loss curves before moving this head to Colab. The Colab run will train the backbone and head together, starting from this selected linear head.
